This is a notebook to visualise the spectroscopy data from the USGS data set

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import re
import glob


In [ ]:
class USGSSatelliteSpectra:
    def __init__(self, base_dir, satellite='ASTER'):
        """
        Initialize the USGS Spectral Library satellite data loader
        
        Parameters:
        -----------
        base_dir : str
            Base directory containing USGS Spectral Library files
        satellite : str
            Satellite sensor name (e.g., 'ASTER', 'LSAT8', 'SNTL2', 'WV3')
        """
        self.base_dir = Path(base_dir)
        self.satellite = satellite
        self.prefix = f"S07{satellite}_"
        
        # Find the appropriate data directory
        self.data_dir = list(self.base_dir.glob(f"**/ASCIIdata_splib07b_rs{satellite}"))[0]
        if not self.data_dir.exists():
            raise FileNotFoundError(f"Could not find data directory for {satellite}")
        
        print(f"Found data directory: {self.data_dir}")
        
        # Initialize collections
        self.spectra = {}
        self.wavelengths = None
        self.bandpass = None
        
        # Load wavelength data
        self._load_wavelength_data()
    
    def _load_wavelength_data(self):
        """Load wavelength data for the satellite sensor"""
        # Find the wavelength file
        wavelength_pattern = f"{self.prefix}Wavelengths_{self.satellite}*microns.txt"
        wavelength_files = list(self.data_dir.glob(wavelength_pattern))
        
        if not wavelength_files:
            raise FileNotFoundError(f"Could not find wavelength file matching pattern: {wavelength_pattern}")
        
        wavelength_file = wavelength_files[0]
        print(f"Loading wavelength data from: {wavelength_file}")
        
        # Load the wavelength data
        self.wavelengths = np.loadtxt(wavelength_file)
        print(f"Loaded wavelength data: {len(self.wavelengths)} bands")
        
        # Try to load bandpass data if available
        bandpass_pattern = f"{self.prefix}Bandpass_{self.satellite}*microns.txt"
        bandpass_files = list(self.data_dir.glob(bandpass_pattern))
        
        if bandpass_files:
            bandpass_file = bandpass_files[0]
            print(f"Loading bandpass data from: {bandpass_file}")
            self.bandpass = np.loadtxt(bandpass_file)
            print(f"Loaded bandpass data: {len(self.bandpass)} bands")
        else:
            print(f"No bandpass data found for {self.satellite}")
    
    def _parse_filename(self, filename):
        """
        Parse the spectrum filename to extract metadata
        
        Example: S07ASTER_Actinolite_HS22.4B_ASDFRb_AREF.txt
        """
        basename = os.path.basename(filename)
        # Remove prefix and file extension
        parts = basename.replace(self.prefix, '').replace('.txt', '').split('_')
        
        if len(parts) < 3:
            return None
        
        # Extract material, sample ID, and measurement info
        material = parts[0]
        
        # Extract spectrometer and purity code
        spec_code = parts[-2]
        spec_match = re.match(r'([A-Z]+[0-9]*)([a-z]+)', spec_code)
        
        if not spec_match:
            return None
            
        spectrometer = spec_match.group(1)
        purity = spec_match.group(2)
        
        # Extract sample ID (everything between material and spectrometer)
        sample_id = '_'.join(parts[1:-2])
        
        # Extract measurement type
        measurement_type = parts[-1]
        
        return {
            'material': material,
            'sample_id': sample_id,
            'spectrometer': spectrometer,
            'purity': purity,
            'measurement_type': measurement_type,
            'filename': basename,
            'full_path': filename,
            'satellite': self.satellite
        }
    
    def load_minerals(self, max_samples=None):
        """
        Load mineral spectra from Chapter M
        
        Parameters:
        -----------
        max_samples : int or None
            Maximum number of samples to load (None for all)
            
        Returns:
        --------
        dict
            Dictionary of loaded mineral spectra
        """
        # Find the minerals directory
        minerals_dir = self.data_dir / "ChapterM_Minerals"
        if not minerals_dir.exists():
            raise FileNotFoundError(f"Could not find minerals directory: {minerals_dir}")
        
        # Find all spectrum files
        spectrum_files = list(minerals_dir.glob(f"{self.prefix}*.txt"))
        
        if max_samples and max_samples < len(spectrum_files):
            spectrum_files = spectrum_files[:max_samples]
        
        print(f"Found {len(spectrum_files)} mineral spectra files")
        
        # Load each spectrum
        for i, filename in enumerate(spectrum_files):
            if i % 100 == 0:
                print(f"Loading spectrum {i+1}/{len(spectrum_files)}")
            
            try:
                # Parse filename for metadata
                metadata = self._parse_filename(str(filename))
                if not metadata:
                    print(f"Could not parse filename: {filename}")
                    continue
                
                # Load spectrum data
                spectrum = np.loadtxt(filename)
                
                # Replace deleted channels with NaN
                spectrum[spectrum < -1e30] = np.nan
                
                # Create a unique key
                key = f"{metadata['material']}_{metadata['sample_id']}"
                
                # Store the data
                self.spectra[key] = {
                    'metadata': metadata,
                    'spectrum': spectrum
                }
                
            except Exception as e:
                print(f"Error loading spectrum {filename}: {str(e)}")
        
        print(f"Successfully loaded {len(self.spectra)} mineral spectra")
        return self.spectra
    
    def plot_spectrum(self, spectrum_key, figsize=(10, 6), ax=None):
        """
        Plot a single spectrum
        
        Parameters:
        -----------
        spectrum_key : str
            Key of the spectrum to plot
        figsize : tuple
            Figure size (ignored if ax is provided)
        ax : matplotlib.axes.Axes
            Axes to plot on (optional)
            
        Returns:
        --------
        matplotlib.axes.Axes
            The axes on which the plot was drawn
        """
        # Check if the spectrum exists
        if spectrum_key not in self.spectra:
            raise KeyError(f"Spectrum key not found: {spectrum_key}")
        
        # Get the spectrum data
        spectrum_data = self.spectra[spectrum_key]
        spectrum = spectrum_data['spectrum']
        metadata = spectrum_data['metadata']
        
        # Create figure if needed
        if ax is None:
            fig, ax = plt.subplots(figsize=figsize)
        
        # Plot the spectrum
        ax.plot(self.wavelengths, spectrum, 'o-', label=spectrum_key)
        
        # Add labels
        ax.set_xlabel('Wavelength (μm)')
        
        if metadata['measurement_type'] == 'AREF':
            ax.set_ylabel('Absolute Reflectance')
        elif metadata['measurement_type'] == 'RREF':
            ax.set_ylabel('Relative Reflectance')
        elif metadata['measurement_type'] == 'TRAN':
            ax.set_ylabel('Transmission')
        else:
            ax.set_ylabel('Value')
        
        ax.set_title(f"{metadata['material']} {metadata['sample_id']} - {self.satellite}")
        ax.grid(True, linestyle='--', alpha=0.7)
        
        # If bandpass data is available, add vertical bars for band centers
        if self.bandpass is not None:
            for i, (wl, bp) in enumerate(zip(self.wavelengths, self.bandpass)):
                # Plot vertical line at band center
                ax.axvline(x=wl, color='gray', linestyle='--', alpha=0.5)
                # Annotate band number
                ax.text(wl, ax.get_ylim()[1] * 0.95, f"B{i+1}", 
                        horizontalalignment='center', fontsize=8)
        
        return ax
    
    def compare_spectra(self, keys, figsize=(12, 8), title=None):
        """
        Compare multiple spectra on a single plot
        
        Parameters:
        -----------
        keys : list
            List of spectrum keys to compare
        figsize : tuple
            Figure size
        title : str
            Plot title (optional)
            
        Returns:
        --------
        matplotlib.figure.Figure
            The figure object
        """
        fig, ax = plt.subplots(figsize=figsize)
        
        for key in keys:
            if key in self.spectra:
                spectrum = self.spectra[key]['spectrum']
                metadata = self.spectra[key]['metadata']
                
                # Plot the spectrum
                ax.plot(self.wavelengths, spectrum, 'o-', label=f"{metadata['material']} {metadata['sample_id']}")
        
        # Add labels
        ax.set_xlabel('Wavelength (μm)')
        ax.set_ylabel('Reflectance / Transmission')
        
        if title:
            ax.set_title(title)
        else:
            ax.set_title(f"{self.satellite} Spectral Comparison")
        
        ax.grid(True, linestyle='--', alpha=0.7)
        ax.legend()
        
        plt.tight_layout()
        return fig
    
    def plot_mineral_family(self, mineral_family, max_samples=10, figsize=(12, 8)):
        """
        Plot spectra for a mineral family
        
        Parameters:
        -----------
        mineral_family : str
            Partial name of the mineral family to plot
        max_samples : int
            Maximum number of samples to include
        figsize : tuple
            Figure size
            
        Returns:
        --------
        matplotlib.figure.Figure
            The figure object
        """
        # Find matching spectra
        matching_keys = [key for key in self.spectra.keys() 
                         if mineral_family.lower() in key.lower()]
        
        # Limit the number of samples
        if max_samples and max_samples < len(matching_keys):
            matching_keys = matching_keys[:max_samples]
        
        if not matching_keys:
            print(f"No spectra found matching mineral family: {mineral_family}")
            return None
        
        print(f"Found {len(matching_keys)} matching spectra for {mineral_family}")
        
        # Plot the spectra
        return self.compare_spectra(matching_keys, figsize=figsize, 
                                   title=f"{self.satellite} {mineral_family.title()} Family Spectra")
    
    def plot_band_combinations(self, spectrum_key, band_combinations=None, figsize=(12, 10)):
        """
        Plot color composite visualizations for different band combinations
        
        Parameters:
        -----------
        spectrum_key : str
            Key of the spectrum to analyze
        band_combinations : list of tuples
            List of (R,G,B) band combinations to visualize
        figsize : tuple
            Figure size
            
        Returns:
        --------
        matplotlib.figure.Figure
            The figure object
        """
        # Check if the spectrum exists
        if spectrum_key not in self.spectra:
            raise KeyError(f"Spectrum key not found: {spectrum_key}")
        
        # Get the spectrum data
        spectrum_data = self.spectra[spectrum_key]
        spectrum = spectrum_data['spectrum']
        metadata = spectrum_data['metadata']
        
        # Default band combinations based on satellite
        if band_combinations is None:
            if self.satellite == 'ASTER':
                band_combinations = [
                    (2, 1, 0),  # Visible RGB (bands 3,2,1)
                    (4, 3, 2),  # SWIR composite
                    (4, 5, 8),  # Clay minerals
                    (6, 2, 1)   # Vegetation and iron
                ]
            elif self.satellite == 'LSAT8':
                band_combinations = [
                    (2, 1, 0),  # Natural color
                    (3, 2, 1),  # False color
                    (5, 4, 3),  # SWIR
                    (6, 5, 2)   # Geology
                ]
            elif self.satellite == 'SNTL2':
                band_combinations = [
                    (2, 1, 0),  # Natural color
                    (3, 2, 1),  # False color
                    (7, 3, 2),  # SWIR
                    (11, 8, 2)  # Geology
                ]
            else:
                # Generic combinations if satellite not recognized
                num_bands = len(self.wavelengths)
                band_combinations = [
                    (min(2, num_bands-1), min(1, num_bands-1), 0),
                    (min(4, num_bands-1), min(3, num_bands-1), min(2, num_bands-1)),
                    (min(6, num_bands-1), min(4, num_bands-1), min(2, num_bands-1))
                ]
        
        # Create figure
        num_combinations = len(band_combinations)
        fig, axs = plt.subplots(num_combinations, 1, figsize=figsize)
        if num_combinations == 1:
            axs = [axs]
        
        # Plot each band combination
        for i, (r, g, b) in enumerate(band_combinations):
            # Check if bands are within range
            max_band_idx = len(self.wavelengths) - 1
            r = min(r, max_band_idx)
            g = min(g, max_band_idx)
            b = min(b, max_band_idx)
            
            # Create a bar plot for the 3 bands
            band_values = [spectrum[b], spectrum[g], spectrum[r]]
            band_labels = [f'B{b+1} ({self.wavelengths[b]:.3f}μm)', 
                          f'B{g+1} ({self.wavelengths[g]:.3f}μm)', 
                          f'B{r+1} ({self.wavelengths[r]:.3f}μm)']
            
            bars = axs[i].bar(['Blue', 'Green', 'Red'], band_values, color=['blue', 'green', 'red'])
            
            # Add the actual band numbers and wavelengths
            for bar, label in zip(bars, band_labels):
                axs[i].text(bar.get_x() + bar.get_width()/2, 0.02, 
                          label, ha='center', va='bottom', 
                          rotation=45, fontsize=10)
            
            # Add title
            axs[i].set_title(f"Band Combination R:{r+1} G:{g+1} B:{b+1}")
            
            # Add band ratio if applicable
            if r != b:
                ratio = spectrum[r] / spectrum[b] if spectrum[b] != 0 else np.nan
                axs[i].text(0.98, 0.95, f"Band Ratio R/B: {ratio:.3f}", 
                           transform=axs[i].transAxes, ha='right', va='top',
                           bbox=dict(facecolor='white', alpha=0.8))
        
        plt.tight_layout()
        plt.suptitle(f"{metadata['material']} {metadata['sample_id']} - {self.satellite} Band Values", 
                    fontsize=14, y=1.02)
        
        return fig
    
    def export_to_dataframe(self):
        """
        Export the spectra to a pandas DataFrame
        
        Returns:
        --------
        pandas.DataFrame
            DataFrame containing the spectra and metadata
        """
        # Create list to hold records
        records = []
        
        for key, data in self.spectra.items():
            # Extract metadata
            metadata = data['metadata']
            spectrum = data['spectrum']
            
            # Create record
            record = {
                'key': key,
                'material': metadata['material'],
                'sample_id': metadata['sample_id'],
                'spectrometer': metadata['spectrometer'],
                'purity': metadata['purity'],
                'measurement_type': metadata['measurement_type'],
                'satellite': metadata['satellite']
            }
            
            # Add spectrum values as individual columns
            for i, wl in enumerate(self.wavelengths):
                record[f'band_{i+1}_{wl:.3f}um'] = spectrum[i]
                
            records.append(record)
        
        # Create DataFrame
        df = pd.DataFrame(records)
        return df
    
    def get_mineral_groups(self):
        """
        Group spectra by mineral types
        
        Returns:
        --------
        dict
            Dictionary with mineral groups and counts
        """
        # Extract mineral types (first part of material name)
        mineral_types = {}
        
        for key, data in self.spectra.items():
            material = data['metadata']['material']
            
            # Extract base mineral name (remove suffixes like numbers or chemical formulas)
            base_mineral = re.sub(r'[0-9._]+$', '', material)
            
            if base_mineral not in mineral_types:
                mineral_types[base_mineral] = []
                
            mineral_types[base_mineral].append(key)
        
        # Sort by number of samples
        sorted_minerals = {k: mineral_types[k] for k in 
                          sorted(mineral_types.keys(), 
                                key=lambda x: len(mineral_types[x]), 
                                reverse=True)}
        
        return sorted_minerals
    
    def spectral_angle_mapper(self, spectrum1, spectrum2):
        """
        Calculate the Spectral Angle Mapper (SAM) between two spectra
        
        Parameters:
        -----------
        spectrum1, spectrum2 : array-like
            The two spectra to compare
            
        Returns:
        --------
        float
            Spectral angle in degrees
        """
        # Remove NaN values
        valid = ~np.isnan(spectrum1) & ~np.isnan(spectrum2)
        if np.sum(valid) < 2:
            return np.nan
            
        # Get valid values
        s1 = spectrum1[valid]
        s2 = spectrum2[valid]
        
        # Normalize
        n1 = s1 / np.sqrt(np.sum(s1**2))
        n2 = s2 / np.sqrt(np.sum(s2**2))
        
        # Calculate angle
        angle = np.arccos(np.clip(np.sum(n1 * n2), -1.0, 1.0))
        return angle * (180.0 / np.pi)  # Convert to degrees
    
    def find_similar_spectra(self, target_key, max_results=10):
        """
        Find spectra similar to a target spectrum using spectral angle mapping
        
        Parameters:
        -----------
        target_key : str
            Key of the target spectrum
        max_results : int
            Maximum number of similar spectra to return
            
        Returns:
        --------
        dict
            Dictionary with spectrum keys and spectral angles
        """
        # Check if the target spectrum exists
        if target_key not in self.spectra:
            raise KeyError(f"Target spectrum key not found: {target_key}")
        
        # Get the target spectrum
        target_spectrum = self.spectra[target_key]['spectrum']
        
        # Calculate spectral angle for all spectra
        results = {}
        for key, data in self.spectra.items():
            if key != target_key:
                spectrum = data['spectrum']
                angle = self.spectral_angle_mapper(target_spectrum, spectrum)
                results[key] = angle
        
        # Sort by angle (smaller is more similar)
        sorted_results = {k: results[k] for k in 
                         sorted(results.keys(), 
                               key=lambda x: results[x])}
        
        # Return top results
        top_results = {k: sorted_results[k] for k in list(sorted_results.keys())[:max_results]}
        return top_results
    
    def plot_similar_spectra(self, target_key, max_results=5, figsize=(14, 10)):
        """
        Find and plot spectra similar to a target spectrum
        
        Parameters:
        -----------
        target_key : str
            Key of the target spectrum
        max_results : int
            Maximum number of similar spectra to display
        figsize : tuple
            Figure size
            
        Returns:
        --------
        matplotlib.figure.Figure
            The figure object
        """
        # Find similar spectra
        similar_spectra = self.find_similar_spectra(target_key, max_results)
        
        # Create figure
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize)
        
        # Plot target spectrum
        target_spectrum = self.spectra[target_key]['spectrum']
        target_metadata = self.spectra[target_key]['metadata']
        ax1.plot(self.wavelengths, target_spectrum, 'b-', linewidth=2.5,
                label=f"Target: {target_metadata['material']} {target_metadata['sample_id']}")
        
        # Plot similar spectra
        for i, (key, angle) in enumerate(similar_spectra.items()):
            spectrum = self.spectra[key]['spectrum']
            metadata = self.spectra[key]['metadata']
            
            # Plot with varying colors
            color = plt.cm.jet(i / max_results)
            ax1.plot(self.wavelengths, spectrum, color=color, linestyle='-', alpha=0.7,
                    label=f"{metadata['material']} {metadata['sample_id']} (SAM: {angle:.2f}°)")
        
        ax1.set_xlabel('Wavelength (μm)')
        ax1.set_ylabel('Reflectance')
        ax1.set_title('Similar Spectra Comparison')
        ax1.grid(True, linestyle='--', alpha=0.7)
        ax1.legend()
        
        # Plot spectral angles as bar chart
        keys = list(similar_spectra.keys())
        angles = list(similar_spectra.values())
        labels = [f"{self.spectra[k]['metadata']['material']} {self.spectra[k]['metadata']['sample_id']}" 
                 for k in keys]
        
        bars = ax2.bar(range(len(keys)), angles)
        
        # Add mineral names
        ax2.set_xticks(range(len(keys)))
        ax2.set_xticklabels(labels, rotation=45, ha='right')
        
        # Add spectral angle values
        for i, v in enumerate(angles):
            ax2.text(i, v + 0.5, f"{v:.2f}°", ha='center')
        
        ax2.set_ylabel('Spectral Angle (degrees)')
        ax2.set_title('Spectral Angle Comparison (lower is more similar)')
        ax2.grid(True, axis='y', linestyle='--', alpha=0.7)
        
        plt.tight_layout()
        plt.subplots_adjust(hspace=0.3)
        
        return fig